Autodecoder


In [ ]:
import sys
sys.path.append('../')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import json
from tqdm import tqdm, trange
import os

from src.models.vae import VAE
from src.utils.datasets import get_bin_mnist_loaders, get_gpu_data
from src.training.train_epoch import train_epoch, evaluate_epoch
from src.training.train_vae import train_beta_vae
from src.utils.plotting import visualize_reconstructions, visualize_generations, plot_rd_curve


from torch.utils.data import DataLoader


# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

"""
Main training loop for multiple beta values
"""
#A100 on Google Colab
device = "cuda" if torch.cuda.is_available() else "cpu"

# Configuration - reduced for faster experimentation
beta_values = [0.01, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 3.0, 4.0, 5.0, 10.0, 20.0]
latent_dim = 64
epochs = 75  # Reduced for faster results
lr = 1e-3
batch_size = 2048
data_path = "src/data/"
output_dir = "src/results/"

results = {}
models = {}

# Train models with different beta values
print("TRAINING: Exploring RD")
print("-" * 60)
for beta in beta_values:
    R, D, model = train_beta_vae(
        beta=beta,
        latent_dim=latent_dim,
        epochs=epochs,
        lr=lr,
        device=device,
        data_path=data_path,
        batch_size=batch_size,
    )
    results[beta] = {"R": R, "D": D}
    models[beta] = model
    # Save model checkpoint
    torch.save(
      {
          "beta": beta,
            "model_state_dict": model.state_dict(),
            "R": R,
            "D": D,
        },
        f"{output_dir}/model_beta_{beta}.pt",
    )

# Plot Rate-Distortion curve
print("\nplots")
print("-" * 60)

plot_rd_curve(results, save_path=f"{output_dir}/rd_curve.png")

# Load test data for visualization
_, _, test_loader = get_bin_mnist_loaders(128)
# Visualize reconstructions and generations
viz_betas = [0.1, 0.5, 1.0, 5.0, 10.0]
viz_models = {b: models[b] for b in viz_betas}
visualize_reconstructions(
    viz_models,
    test_loader,
    viz_betas,
    device,
    save_path=f"{output_dir}/reconstructions.png",
)
visualize_generations(
    viz_models, viz_betas, device, save_path=f"{output_dir}/generations.png"
)
